In [1]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('../../env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
print(AZURE_OPENAI_API_KEY[:10])
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['MODEL_API_VERSION'] = os.getenv('MODEL_API_VERSION')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'AGENT'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    if len(os.getenv('LANGCHAIN_API_KEY')) > 0:
        print('랭스미스로 추적 중입니다 :', os.getenv('LANGSMITH_API_KEY')[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

3BHTZdVIXx
gpt-5-nano
랭스미스로 추적 중입니다 : lsv2_pt_ef


In [2]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    model=os.environ['MODEL_NAME'],
    azure_deployment = os.environ["MODEL_NAME"],
    azure_endpoint = os.environ["END_POINT"],
    openai_api_version = os.environ["MODEL_API_VERSION"],
    openai_api_key = os.environ["AZURE_OPENAI_API_KEY"],
)

In [3]:
import os
WEATHER_URL = os.getenv("WEATHER_URL", "http://localhost:9101")
OUTFIT_URL  = os.getenv("OUTFIT_URL",  "http://localhost:9102")
base_urls = [WEATHER_URL, OUTFIT_URL]

In [4]:
import subprocess
!fuser -k 8001/tcp
!fuser -k 9101/tcp
!fuser -k 9102/tcp

import sys, subprocess
print("using python:", sys.executable)
subprocess.Popen([sys.executable, "mcp_openmeteo_server.py"])
subprocess.Popen([sys.executable, "a2a_weather_agent.py"])
subprocess.Popen([sys.executable, "a2a_outfit_recom_agent.py"])

using python: /anaconda/envs/kt_env/bin/python


<Popen: returncode: None args: ['/anaconda/envs/kt_env/bin/python', 'a2a_out...>

In [5]:
from a2a_tools import get_agent_card, call_agent

In [7]:
# 1) 에이전트 카드 조회 및 출력
weather_card_json = await get_agent_card.ainvoke({"base_url": WEATHER_URL})
outfit_card_json = await get_agent_card.ainvoke({"base_url": OUTFIT_URL})
print("=== Weather Agent Card ===\n" + weather_card_json)
print("=== Outfit Agent Card ===\n" + outfit_card_json)

INFO:     127.0.0.1:59562 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:56860 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
=== Weather Agent Card ===
{
  "additionalInterfaces": null,
  "capabilities": {
    "extensions": null,
    "pushNotifications": null,
    "stateTransitionHistory": null,
    "streaming": false
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "description": "입력 받은 도시의 날씨를 요약",
  "documentationUrl": null,
  "iconUrl": null,
  "name": "WeatherSummary",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "provider": null,
  "security": null,
  "securitySchemes": null,
  "signatures": null,
  "skills": [
    {
      "description": "입력 받은 도시의 날씨를 불릿 리스트 형태로 요약",
      "examples": [
        "서울 날씨 요약 부탁",
        "오늘 날씨 알려줘",
        "내일 부산 날씨"
      ],
      "id": "weather_summary",
      "inputModes": null,
      "name": "weather_summary",
      "outputModes": null,
      "secur

In [8]:
# 2) 파이프라인: 날씨 요약 → 옷 추천
print("=== [Weather Agent]에게 날씨 정보를 요청 합니다.===")
weather_text = await call_agent.ainvoke({"base_url": WEATHER_URL, "user_text": "오늘 서울의 날씨"})
print("\n[날씨 요약]\n", weather_text)
print("\n----------------------------------------------------------\n")

=== [Weather Agent]에게 날씨 정보를 요청 합니다.===
INFO:     127.0.0.1:59568 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:43120 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:43128 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:43122 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:43138 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:43142 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:43156 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:43172 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:43174 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:43176 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:44604 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:44612 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59582 - "POST / HTTP/1.1" 200 OK

[날씨 요약]
 - 도시: Seoul
- 날짜: 2026-02-27 (오늘)
- 최저 기온 / 최고 기온: 3.2°C / 12.6°C
- 강수량: 0.0 mm
- 일일 구름량(평균): 27.6%
- 자외선 지수: 최대 4.7
- 평균 풍속: 4.6 m/s
- 시간대별 주요 특징
  - 00:00~04:00: 구름 많음, 구름량 66%~91%
  - 0

In [9]:
print("=== [Outfit Agent]에게 옷 추천을 요청 합니다.===")
outfit_text = await call_agent.ainvoke({"base_url": OUTFIT_URL, "user_text": weather_text or "서울 날씨 요약"})
print("\n[옷 추천]\n", outfit_text)
print("\n----------------------------------------------------------\n")

=== [Outfit Agent]에게 옷 추천을 요청 합니다.===
INFO:     127.0.0.1:55032 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:55040 - "POST / HTTP/1.1" 200 OK

[옷 추천]
 [추천 의복/소지품] 오늘 서울 날씨 요약
- 최저/최고 기온: 약 3°C → 13°C 내외 (주간 상승, 오후 다소 따뜻)
- 강수확률: 0%로 비/눈 없음
- 구름: 대체로 맑음~부분적으로 구름 많음
- 자외선 지수: 4.7으로 보통 수준, 오후에 햇빛 조금 강함
- 바람: 약 4.6 m/s로 바람은 비교적 잔잔하지만 오후에는 체감 온도 더 떨어질 수 있음(특히 바람 많은 구간)
- 변동 포인트: 오전은 다소 흐리고 쌀쌀, 낮에는 맑아 따뜻하게 느껴질 가능성 큼

오늘의 의복 추천 (하루 종일 잘 맞는 레이어링)
- 기본 컨셉: 쌀쌀한 아침을 견디면서도 낮에는 가볍게 벗어 입을 수 있는 3단 레이어링
  - 바닥: 편안한 면/울 혼방 바지, 또는 보온이 되는 두꺼운 청바지
  - 중간 레이어: 얇은 니트 또는 경량 플리스
  - 겉 레이어: 바람막이/경량 패딩 혹은 트렌치 코트처럼 가볍고 바람을 막아주는 자켓
  - 신발: 보온이 되고 미끄럼 방지 가능한 로퍼/스니커즈 혹은 경량 부츠
  - 액세서리: 손목/목 보온을 위한 얇은 목도리, 필요 시 얇은 비니 또는 모자

- 시간대별 추천 레이어링 팁
  - 00:00–04:00 (다소 흐림, 기온 낮음): 바람막이 위에 두꺼운 슬리브가 있는 상의 + 니트/플리스 + 기온 낮으므로 얇은 모자나 얇은 목도리 권장
  - 09:00–11:00 (구름은 많지 않지만 차가움): 중간 레이어를 자주 조절 가능하게 입되, 바람막이로 체감 온도 조절
  - 12:00–13:00 (맑음, 강한 직사광선은 아직 없음): 선글라스와 자외선 차단제 준비, 경량 자켓 벗어도 무방할 정도로 온도 상승
  - 14:00

## 다른 Agent를 능동적으로 선택하는 Agent

In [10]:
from a2a_tools import list_agents, call_agent, get_agent_card

tools = [list_agents, get_agent_card, call_agent]
tools

[StructuredTool(name='list_agents', description='여러 base_url의 에이전트 카드를 조회해, 이름·설명·스킬 요약을 반환합니다. LLM이 호출할 에이전트를 고를 때 사용합니다.', args_schema=<class 'langchain_core.utils.pydantic.list_agents'>, coroutine=<function list_agents at 0x763d00c08e00>),
 StructuredTool(name='get_agent_card', description='주어진 URL에서 에이전트 카드(JSON)를 가져와 반환합니다.', args_schema=<class 'langchain_core.utils.pydantic.get_agent_card'>, coroutine=<function get_agent_card at 0x763d00c08680>),
 StructuredTool(name='call_agent', description='지정한 A2A 에이전트에 user_text를 보내고, 응답 텍스트를 반환합니다.', args_schema=<class 'langchain_core.utils.pydantic.call_agent'>, coroutine=<function call_agent at 0x763d00c094e0>)]

In [11]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
SYSTEM = f"""너는 A2A 형식의 멀티 에이전트에서 오케스트레이터 역할을 하는 에이전트다.

네 임무는 두 단계로 구성된다.
첫 번째: '사용자의 목표'를 달성하기 위해 사용 가능한 A2A 에이전트들을 탐색한다.
두 번째: 적절한 순서로 에이전트를 호출하여 최종 답을 만들어내는 것이다.

원칙:
1) 먼저 list_agents 도구로 이용 가능한 에이전트와 그들의 'name, description, skills'을 확인한다. 확인된 에이전트 정보를 문장으로 정리하여 반환한다.
2) 필요하다면 get_agent_card로 특정 에이전트의 세부 Capabilities를 본다.
3) 목적 달성을 위해 필요한 에이전트들의 호출 순서를 계획한다.
4) 각 단계에서 call_agent 도구를 사용하여 실제 호출을 수행한다.
   - 이전 단계의 출력은 다음 단계의 입력으로 넘긴다.
5) 최종적으로 사용자에게 깔끔한 한국어로 결과만 전달한다.
6) 마지막에는 핵심 요소들만 정리하여 덧붙인다.

에이전트를 호출하기 위한 base_url은 다음과 같다. : {base_urls}
"""

agent = create_agent(llm, tools, system_prompt=SYSTEM,    checkpointer=InMemorySaver())
        

In [12]:
async def run_agent(query, user_id, mode = 'invoke'):
    if mode == 'invoke':
        result =  await agent.ainvoke({"messages": [{"role": "user", "content": query}]}, config={"configurable": {"thread_id": user_id}})
        return result['messages'][-1].content
    else:
        async for chunk in agent.astream(
            {"messages": [{"role": "user", "content": query}]},
            config={"configurable": {"thread_id": user_id}},
            stream_mode="values",
        ):
            msgs = chunk.get("messages")
            if msgs:
                print(msgs[-1].pretty_repr(html=True))


In [13]:
await run_agent("오늘 서울의 날씨만 알려줘", '1', mode='stream')

================================ Human Message =================================

오늘 서울의 날씨만 알려줘
================================== Ai Message ==================================
Tool Calls:
  list_agents (call_7YIizwYiIU65ReOXDH67J5on)
 Call ID: call_7YIizwYiIU65ReOXDH67J5on
  Args:
    base_urls: ['http://localhost:9101', 'http://localhost:9102']
INFO:     127.0.0.1:33474 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:45068 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
================================= Tool Message =================================
Name: list_agents

[
  {
    "base_url": "http://localhost:9101",
    "summary": "Name: WeatherSummary\nDescription: 입력 받은 도시의 날씨를 요약\nSkills:\n- weather_summary: 입력 받은 도시의 날씨를 불릿 리스트 형태로 요약"
  },
  {
    "base_url": "http://localhost:9102",
    "summary": "Name: OutfitRecommender\nDescription: 요약된 날씨 문장을 받아 그날 챙길 의복/소지품을 추천합니다.\nSkills:\n- outfit_from_weather: 날씨 문장을 입력 받아 그 날 필요한 의복/소지품을 추천"
  }
]
========

In [ ]:
await run_agent("뭘 입으면 좋을까?", '1',mode='stream')

In [ ]:
await run_agent("날씨를 알고 싶을 때 내가 쓸 수 있는 에이전트는 뭐야?", '1', mode='stream')